<a href="https://colab.research.google.com/github/angelinetipa/deped-data-audit/blob/main/deped_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ================================================================
# MOUNT GOOGLE DRIVE — allows Colab to read/write files in Drive
# so data and outputs persist after the session ends.
# Running this will prompt a Google sign-in/permission request —
# click "Connect to Google Drive" and grant access.
# ================================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
# Path points inside the mounted Google Drive, not a local upload.
# Update the filename if the file has been renamed.
RAW = "/content/drive/MyDrive/deped-audit/data/raw.xlsx"

import pandas as pd
df = pd.read_excel(RAW, sheet_name="DB", header=4)
print(df.shape)  # quick check: expected output (60167, 72)

(60167, 72)


In [8]:
# Auto-discovery, Part A (v4): split into two buckets, no stopword
# filtering. Bucket 2 will include some real short words alongside
# actual abbreviations; these are reviewed manually rather than
# filtered automatically.

import re
from collections import Counter

# Bucket 1: tokens containing "." or "/". Punctuation is treated as
# a structural signal of an abbreviation (e.g. "St.", "E/S"), and
# produces very few false positives.
def find_punctuated(series, max_letters=5, min_count=5):
    tokens = Counter()
    for text in series.dropna().astype(str):
        for word in text.split():
            has_mark = "." in word or "/" in word
            letters_only = re.sub(r"[^A-Za-z]", "", word)
            if has_mark and 1 <= len(letters_only) <= max_letters:
                tokens[word] += 1
    # Only patterns meeting the minimum count are kept, so one-off
    # typos are separated from recurring patterns.
    return Counter({w: c for w, c in tokens.items() if c >= min_count})

# Bucket 2: short ALL-CAPS tokens with no punctuation (e.g. "ES",
# "NHS"). No stopword list is applied, so some real short words
# (e.g. roman numerals, "SAN") will also appear here and require
# a manual pass to separate from genuine abbreviations.
def find_caps_no_punct(series, max_letters=5, min_count=5):
    tokens = Counter()
    for text in series.dropna().astype(str):
        for word in text.split():
            clean = re.sub(r"[^A-Za-z]", "", word)
            if (
                word.isupper()
                and "." not in word and "/" not in word
                and 1 <= len(clean) <= max_letters
            ):
                tokens[word] += 1
    return Counter({w: c for w, c in tokens.items() if c >= min_count})

# Bucket 1 result for School Name. Expected to surface items such
# as "St.", "Inc.", "E/S", "P/S".
print("School Name — punctuation-marked abbreviations")
for word, count in find_punctuated(df["School Name"]).most_common(30):
    print(f"{word}: {count}")

# Bucket 2 result for School Name. Expected to surface items such
# as "ES", "NHS", "CS", "CES", alongside some real short words that
# require manual filtering.
print("\nSchool Name — ALL-CAPS short tokens (manual review)")
for word, count in find_caps_no_punct(df["School Name"]).most_common(30):
    print(f"{word}: {count}")

# Bucket 1 result for Street Address. Expected to surface items
# such as "Brgy.", "St.", "Sta.", "Subd.".
print("\nStreet Address — punctuation-marked abbreviations")
for word, count in find_punctuated(df["Street Address"]).most_common(30):
    print(f"{word}: {count}")

# Bucket 2 result for Street Address. Expected to surface items
# such as "PUROK", "SITIO", "SAN", alongside some real short words
# that require manual filtering.
print("\nStreet Address — ALL-CAPS short tokens (manual review)")
for word, count in find_caps_no_punct(df["Street Address"]).most_common(30):
    print(f"{word}: {count}")

School Name — punctuation-marked abbreviations
Inc.: 5826
St.: 963
Sta.: 783
Sr.: 443
Sto.: 421
INC.: 387
A.: 229
M.: 194
P.: 193
C.: 175
B.: 167
L.: 155
Elem.: 142
R.: 138
E.: 125
F.: 125
G.: 120
Dr.: 118
S.: 116
T.: 115
Mem.: 104
D.: 102
V.: 99
E/S: 87
Gen.: 76
Sch.: 74
Ext.: 70
J.: 66
Mt.: 50
H.: 48

School Name — ALL-CAPS short tokens (manual review)
ES: 16692
NHS: 2088
PS: 1397
CS: 440
CES: 435
HS: 343
HIGH: 252
II: 238
I: 233
OF: 177
MES: 176
IS: 175
ES): 164
MS: 131
SPED: 123
SDA: 82
NHS): 67
MNHS: 63
STI: 63
UCCP: 60
AMA: 52
INC: 51
SAN: 48
SHS: 43
III: 40
IP: 40
AND: 39
MHS: 29
A: 28
PS): 27

Street Address — punctuation-marked abbreviations
Brgy.: 4161
St.: 2762
St.,: 1579
Sta.: 805
Sto.: 407
Subd.: 275
P.: 256
ST.: 238
BRGY.: 227
Prk.: 227
A.: 208
Ave.,: 204
Subd.,: 189
Ave.: 175
Gen.: 173
M.: 166
-Brgy.: 160
J.: 157
Blk.: 150
F.: 148
-n/a: 145
st.: 141
Sor.: 136
Bldg.,: 133
So.: 132
E.: 131
cor.: 123
Rd.: 117
Bgy.: 116
Pob.: 110

Street Address — ALL-CAPS short tokens (manu